In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyApplication").getOrCreate()

In [2]:
df = spark.read.csv("employee-dataset/employee-data.csv", header = True, inferSchema=True)
df.show(10)

+-------+---+------+----------+
|   name|age|salary|department|
+-------+---+------+----------+
|  Alice| 25|  5000|        HR|
|    Bob| 30|  6000|        IT|
|Charlie| 35|  7000|   Finance|
|  David| 28|  5500|     Sales|
|    Eve| 32|  6200|        IT|
|  Frank| 40|  7500|   Finance|
|  Grace| 27|  5300|        HR|
|   Hank| 33|  6400|        IT|
|    Ivy| 29|  5700|     Sales|
|   Jack| 38|  7200|   Finance|
+-------+---+------+----------+
only showing top 10 rows



In [6]:
from pyspark.sql import SparkSession, functions as F

try:
    spark.stop()
except:
    pass
    
# --------------------------------------------------
# 1️⃣ Spark setup
# --------------------------------------------------
spark = SparkSession.builder \
    .appName("Skew_vs_Salting_Demo") \
    .getOrCreate()

# Reduce partitions to exaggerate skew
spark.conf.set("spark.sql.shuffle.partitions", 16)

# --------------------------------------------------
# 2️⃣ Create skewed dataset
# --------------------------------------------------
data = [("A", i) for i in range(900000)] + \
       [("B", i) for i in range(50000)] + \
       [("C", i) for i in range(50000)]

df = spark.createDataFrame(data, ["key", "value"])


In [12]:
data[:10]

[('A', 0),
 ('A', 1),
 ('A', 2),
 ('A', 3),
 ('A', 4),
 ('A', 5),
 ('A', 6),
 ('A', 7),
 ('A', 8),
 ('A', 9)]

In [4]:
df.show(10)
print("Total rows:", df.count())

+---+-----+
|key|value|
+---+-----+
|  A|    0|
|  A|    1|
|  A|    2|
|  A|    3|
|  A|    4|
|  A|    5|
|  A|    6|
|  A|    7|
|  A|    8|
|  A|    9|
+---+-----+
only showing top 10 rows

Total rows: 1000000


In [7]:
# --------------------------------------------------
# 3️⃣ WITHOUT SALTING (Skew problem)
# --------------------------------------------------
print("\n=== WITHOUT SALTING ===")

df_grouped = df.groupBy("key").count()

print("Final result:")
df_grouped.show()

print("Partition distribution AFTER shuffle:")
df_grouped.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .show()




=== WITHOUT SALTING ===
Final result:
+---+------+
|key| count|
+---+------+
|  A|900000|
|  B| 50000|
|  C| 50000|
+---+------+

Partition distribution AFTER shuffle:
+---+-----+
|pid|count|
+---+-----+
|  0|    3|
+---+-----+



In [13]:
df_grouped.withColumn("pid", F.spark_partition_id()) \
    .show()

+---+------+---+
|key| count|pid|
+---+------+---+
|  A|900000|  0|
|  B| 50000|  0|
|  C| 50000|  0|
+---+------+---+



In [8]:
# --------------------------------------------------
# 4️⃣ ADD SALT COLUMN
# --------------------------------------------------
print("\n=== ADDING SALT ===")

salt_buckets = 10

df_salted = (
    df
    .withColumn("salt", (F.rand() * salt_buckets).cast("int"))
    .withColumn("salted_key", F.concat_ws("_", "key", "salt"))
)

print("Sample salted rows:")
df_salted.select("key", "value", "salt", "salted_key").show(10, False)


=== ADDING SALT ===
Sample salted rows:
+---+-----+----+----------+
|key|value|salt|salted_key|
+---+-----+----+----------+
|A  |0    |5   |A_5       |
|A  |1    |3   |A_3       |
|A  |2    |4   |A_4       |
|A  |3    |9   |A_9       |
|A  |4    |1   |A_1       |
|A  |5    |1   |A_1       |
|A  |6    |7   |A_7       |
|A  |7    |5   |A_5       |
|A  |8    |2   |A_2       |
|A  |9    |7   |A_7       |
+---+-----+----+----------+
only showing top 10 rows



In [17]:
# --------------------------------------------------
# 5️⃣ PARTIAL AGGREGATION (Distributed)
# --------------------------------------------------
print("\n=== WITH SALTING (PARTIAL AGG) ===")

partial = df_salted.groupBy("salted_key").count()

print("Partition distribution AFTER salting:")
partial.withColumn("pid", F.spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .show()

print("Sample partial aggregation:")
#partial.orderBy("salted_key").show(20, False)


=== WITH SALTING (PARTIAL AGG) ===
Partition distribution AFTER salting:
+---+-----+
|pid|count|
+---+-----+
|  0|   30|
+---+-----+

Sample partial aggregation:


In [14]:
partial.withColumn("pid", F.spark_partition_id()) \
    .show()

+----------+-----+---+
|salted_key|count|pid|
+----------+-----+---+
|       A_3|89781|  0|
|       A_9|89811|  0|
|       A_4|90656|  0|
|       A_5|89663|  0|
|       A_7|89902|  0|
|       A_0|89600|  0|
|       A_8|90338|  0|
|       A_1|90209|  0|
|       A_2|89823|  0|
|       A_6|90217|  0|
|       B_5| 5015|  0|
|       B_8| 4966|  0|
|       B_3| 4945|  0|
|       B_2| 5082|  0|
|       B_1| 5036|  0|
|       B_9| 4954|  0|
|       B_6| 5130|  0|
|       B_7| 4987|  0|
|       B_4| 4887|  0|
|       B_0| 4998|  0|
+----------+-----+---+
only showing top 20 rows



In [10]:
# --------------------------------------------------
# 6️⃣ FINAL AGGREGATION (Unsalt)
# --------------------------------------------------
print("\n=== FINAL AGGREGATION (UNSALTING) ===")

final = (
    partial
    .withColumn("key", F.split("salted_key", "_")[0])
    .groupBy("key")
    .agg(F.sum("count").alias("count"))
)

final.show()


=== FINAL AGGREGATION (UNSALTING) ===
+---+------+
|key| count|
+---+------+
|  B| 50000|
|  A|900000|
|  C| 50000|
+---+------+



In [11]:
# --------------------------------------------------
# 7️⃣ OPTIONAL: Compare partition workload directly
# --------------------------------------------------
print("\n=== PARTITION WORKLOAD COMPARISON ===")

def count_in_partition(iterator):
    yield sum(1 for _ in iterator)

print("Without salting workload:")
print(df.groupBy("key").count().rdd.mapPartitions(count_in_partition).collect())

print("With salting workload:")
print(partial.rdd.mapPartitions(count_in_partition).collect())


=== PARTITION WORKLOAD COMPARISON ===
Without salting workload:
[3]
With salting workload:
[30]


In [ ]:
# Without salting:
# One partition does most of the work (because of "A")
# Others are underutilized
# With salting:
# "A" split into A_0 ... A_9
# Work evenly distributed across partitions
# Final result still correct after unsalting

<!-- In this script, you first create a deliberately skewed dataset where one key ("A") has far more rows than others, which causes a problem during Spark’s groupBy because all rows with the same key are sent to a single partition, leading to uneven workload and poor performance. To address this, you apply a technique called salting, where you add a random suffix to the key (turning "A" into values like "A_0", "A_1", etc.), effectively spreading the heavy data across multiple partitions so the work can be processed in parallel more evenly. After this first distributed aggregation, you then remove the artificial salt by extracting the original key and aggregating again, which combines the partial results back into the correct final counts. This approach demonstrates how salting helps mitigate skew by redistributing data during processing while still preserving accurate results at the end. -->